# MS MARCO 1M Streaming IVF-PQ / OPQ-IVF-PQ Scale Benchmark

This is **Phase 1.1** of the large-scale retrieval study.

It evaluates a qrels-preserving 1,000,000-passage MS MARCO subset with:

- GPU FlatIP exact retrieval
- GPU IVF-PQ with FP16 lookup tables
- Native Faiss `OPQMatrix` + GPU IVF-PQ
- `nprobe = 8, 16, 32, 64`

The notebook reports strict Recall@10, Success@10, MRR@10, nDCG@10, index build time, serialized deployment-aware storage, P50/P95 GPU search latency, and QPS.

This is a benchmark-only notebook. It does **not** overwrite the deployed FiQA OPQ FastAPI artifact.


In [1]:
# 1. Install dependencies without upgrading Colab's core NumPy / pandas stack

!pip -q install \
  "faiss-gpu-cu12>=1.12.0" \
  "sentence-transformers>=3.0,<4.0" \
  "transformers>=4.40,<5.0" \
  "tqdm" \
  "psutil"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.9/275.9 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 20.4 MB/s eta 0:00:00


In [2]:
# 2. Imports
from __future__ import annotations

import csv
import gc
import json
import math
import os
import random
import tarfile
import time
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import psutil
import torch
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime before continuing.")

print("Torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("Faiss GPUs:", faiss.get_num_gpus())


Torch: 2.11.0+cu128
GPU: Tesla T4
Faiss GPUs: 1


In [3]:
# 3. Configuration

TARGET_PASSAGES = 1_000_000
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
DIMENSION = 384

ENCODE_BATCH_SIZE = 512
ADD_BATCH_SIZE = 50_000
TRAIN_SAMPLE_SIZE = 200_000

NLIST = 4096
M = 96
NBITS = 8
NPROBE_VALUES = [8, 16, 32, 64]
TOP_K = 10
SEARCH_BATCH_SIZE = 64

# Native Faiss OPQMatrix training controls.
OPQ_NITER = 50
OPQ_NITER_PQ = 4

DATA_DIR = Path("msmarco_scale_data")
RAW_DIR = DATA_DIR / "raw"
WORK_DIR = DATA_DIR / f"subset_{TARGET_PASSAGES // 1_000_000}m"
RESULT_DIR = Path("msmarco_scale_results")
for path in (RAW_DIR, WORK_DIR, RESULT_DIR):
    path.mkdir(parents=True, exist_ok=True)

COLLECTION_URL = "https://msmarco.z22.web.core.windows.net/msmarcoranking/collectionandqueries.tar.gz"
QRELS_URL = "https://msmarco.z22.web.core.windows.net/msmarcoranking/qrels.dev.tsv"

print({
    "target_passages": TARGET_PASSAGES,
    "model": EMBEDDING_MODEL,
    "nlist": NLIST,
    "M": M,
    "nbits": NBITS,
    "nprobe_values": NPROBE_VALUES,
    "opq_niter": OPQ_NITER,
})


{'target_passages': 1000000, 'model': 'BAAI/bge-small-en-v1.5', 'nlist': 4096, 'M': 96, 'nbits': 8}


In [4]:
# 4. Download official MS MARCO passage collection and dev qrels

archive_path = RAW_DIR / "collectionandqueries.tar.gz"
qrels_path = RAW_DIR / "qrels.dev.tsv"

if not archive_path.exists():
    !wget -c -O "$archive_path" "$COLLECTION_URL"

if not qrels_path.exists():
    !wget -c -O "$qrels_path" "$QRELS_URL"

# Extract only once.
collection_path = RAW_DIR / "collection.tsv"
queries_path = RAW_DIR / "queries.dev.small.tsv"

if not collection_path.exists() or not queries_path.exists():
    with tarfile.open(archive_path, "r:gz") as tar:
        names = tar.getnames()
        wanted = [
            name for name in names
            if name.endswith("collection.tsv")
            or name.endswith("queries.dev.small.tsv")
        ]
        if not any(name.endswith("collection.tsv") for name in wanted):
            raise FileNotFoundError("collection.tsv was not found in the official archive.")
        if not any(name.endswith("queries.dev.small.tsv") for name in wanted):
            raise FileNotFoundError(
                "queries.dev.small.tsv was not found in the official archive."
            )
        for name in wanted:
            tar.extract(name, RAW_DIR)

        # The archive may unpack into a nested directory.
        nested_collection = next(RAW_DIR.rglob("collection.tsv"))
        nested_queries = next(RAW_DIR.rglob("queries.dev.small.tsv"))
        if nested_collection != collection_path:
            nested_collection.replace(collection_path)
        if nested_queries != queries_path:
            nested_queries.replace(queries_path)

print("Collection:", collection_path, f"{collection_path.stat().st_size / 1e9:.2f} GB")
print("Queries:", queries_path)
print("Qrels:", qrels_path)


--2026-06-28 16:33:37--  https://msmarco.z22.web.core.windows.net/msmarcoranking/collectionandqueries.tar.gz
Resolving msmarco.z22.web.core.windows.net (msmarco.z22.web.core.windows.net)... 57.150.146.100
Connecting to msmarco.z22.web.core.windows.net (msmarco.z22.web.core.windows.net)|57.150.146.100|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1057717952 (1009M) [application/gzip]
Saving to: ‘msmarco_scale_data/raw/collectionandqueries.tar.gz’

msmarco_scale_data/ 100%[===================>]   1009M  41.6MB/s    in 27s     

2026-06-28 16:34:04 (37.0 MB/s) - ‘msmarco_scale_data/raw/collectionandqueries.tar.gz’ saved [1057717952/1057717952]

--2026-06-28 16:34:04--  https://msmarco.z22.web.core.windows.net/msmarcoranking/qrels.dev.tsv
Resolving msmarco.z22.web.core.windows.net (msmarco.z22.web.core.windows.net)... 57.150.146.100
Connecting to msmarco.z22.web.core.windows.net (msmarco.z22.web.core.windows.net)|57.150.146.100|:443... connected.
HTTP request se

/tmp/ipykernel_4352/1087602619.py:31: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(name, RAW_DIR)


Collection: msmarco_scale_data/raw/collection.tsv 3.06 GB
Queries: msmarco_scale_data/raw/queries.dev.small.tsv
Qrels: msmarco_scale_data/raw/qrels.dev.tsv


In [5]:
# 5. Load dev queries/qrels and choose a qrels-preserving 1M subset

def load_tsv_map(path):
    mapping = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            key, value = line.rstrip("\n").split("\t", maxsplit=1)
            mapping[str(key)] = value
    return mapping

def load_qrels(path):
    qrels = {}
    with open(path, "r", encoding="utf-8") as f:
        reader = csv.reader(f, delimiter="\t")
        for row in reader:
            if len(row) < 4:
                continue
            qid, _, pid, rel = row[:4]
            if int(rel) > 0:
                qrels.setdefault(str(qid), set()).add(str(pid))
    return qrels

queries = load_tsv_map(queries_path)
qrels_all = load_qrels(qrels_path)
positive_pids = {pid for positives in qrels_all.values() for pid in positives}

if len(positive_pids) >= TARGET_PASSAGES:
    raise ValueError("Target is smaller than the number of judged-positive passages.")

print("Dev queries:", len(queries))
print("Positive passage IDs retained:", len(positive_pids))


Dev queries: 6980
Positive passage IDs retained: 59096


In [6]:
# 6. First pass: reservoir-sample distractor passage IDs without holding text in RAM

needed_random = TARGET_PASSAGES - len(positive_pids)
reservoir = []
seen_candidates = 0

with open(collection_path, "r", encoding="utf-8") as f:
    for line in tqdm(f, desc="Sampling passage IDs", unit="passages"):
        pid = line.split("\t", 1)[0]
        if pid in positive_pids:
            continue

        seen_candidates += 1
        if len(reservoir) < needed_random:
            reservoir.append(pid)
        else:
            j = random.randint(0, seen_candidates - 1)
            if j < needed_random:
                reservoir[j] = pid

selected_pids = positive_pids | set(reservoir)

if len(selected_pids) != TARGET_PASSAGES:
    raise RuntimeError(
        f"Expected {TARGET_PASSAGES:,} selected passages, got {len(selected_pids):,}."
    )

selected_ids_path = WORK_DIR / "selected_pids.txt"
selected_ids_path.write_text("\n".join(sorted(selected_pids, key=int)), encoding="utf-8")
print("Selected passage IDs:", len(selected_pids))


Sampling passage IDs: 0passages [00:00, ?passages/s]

Selected passage IDs: 1000000


In [7]:
# 7. Second pass: materialize the selected corpus in original collection order

subset_path = WORK_DIR / "collection_subset.tsv"
count = 0

with open(collection_path, "r", encoding="utf-8") as src, open(
    subset_path, "w", encoding="utf-8"
) as dst:
    for line in tqdm(src, desc="Writing selected corpus", unit="passages"):
        pid, passage = line.rstrip("\n").split("\t", maxsplit=1)
        if pid in selected_pids:
            dst.write(f"{pid}\t{passage}\n")
            count += 1

if count != TARGET_PASSAGES:
    raise RuntimeError(f"Subset materialization mismatch: {count:,} != {TARGET_PASSAGES:,}")

# Keep only queries whose judged positives are all available in the selected corpus.
qrels = {
    qid: {pid for pid in pids if pid in selected_pids}
    for qid, pids in qrels_all.items()
}
qrels = {qid: pids for qid, pids in qrels.items() if pids and qid in queries}
eval_qids = sorted(qrels)

print("Subset passages:", count)
print("Evaluation queries:", len(eval_qids))
print("Subset file:", subset_path)


Writing selected corpus: 0passages [00:00, ?passages/s]

Subset passages: 1000000
Evaluation queries: 6980
Subset file: msmarco_scale_data/subset_1m/collection_subset.tsv


In [8]:
# 8. Streaming embedding encoding to disk-backed memmaps

doc_ids_path = WORK_DIR / "doc_ids.int64.memmap"
embeddings_path = WORK_DIR / "embeddings.fp16.memmap"

doc_ids_mm = np.memmap(
    doc_ids_path,
    mode="w+",
    dtype=np.int64,
    shape=(TARGET_PASSAGES,),
)
embeddings_mm = np.memmap(
    embeddings_path,
    mode="w+",
    dtype=np.float16,
    shape=(TARGET_PASSAGES, DIMENSION),
)

model = SentenceTransformer(EMBEDDING_MODEL, device="cuda")

buffer_ids, buffer_texts = [], []
written = 0

def flush_batch():
    global written, buffer_ids, buffer_texts
    if not buffer_texts:
        return
    vecs = model.encode(
        buffer_texts,
        batch_size=ENCODE_BATCH_SIZE,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    ).astype(np.float32)

    if vecs.shape[1] != DIMENSION:
        raise ValueError(f"Unexpected embedding dimension: {vecs.shape[1]}")

    end = written + len(buffer_texts)
    doc_ids_mm[written:end] = np.asarray(buffer_ids, dtype=np.int64)
    embeddings_mm[written:end] = vecs.astype(np.float16)
    written = end
    buffer_ids, buffer_texts = [], []

with open(subset_path, "r", encoding="utf-8") as f:
    for line in tqdm(f, desc="Encoding passages", total=TARGET_PASSAGES):
        pid, passage = line.rstrip("\n").split("\t", maxsplit=1)
        buffer_ids.append(int(pid))
        buffer_texts.append(passage)
        if len(buffer_texts) >= ENCODE_BATCH_SIZE:
            flush_batch()

flush_batch()
doc_ids_mm.flush()
embeddings_mm.flush()

if written != TARGET_PASSAGES:
    raise RuntimeError(f"Embedding count mismatch: {written:,} != {TARGET_PASSAGES:,}")

print("Embedding memmap:", embeddings_path, f"{embeddings_path.stat().st_size / 1e9:.2f} GB")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding passages:   0%|          | 0/1000000 [00:00<?, ?it/s]

Embedding memmap: msmarco_scale_data/subset_1m/embeddings.fp16.memmap 0.77 GB


In [9]:
# 9. Encode evaluation queries and prepare relevance lookup

query_texts = [queries[qid] for qid in eval_qids]
query_vectors = model.encode(
    query_texts,
    batch_size=ENCODE_BATCH_SIZE,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True,
).astype(np.float32)

doc_ids_mm = np.memmap(
    doc_ids_path,
    mode="r",
    dtype=np.int64,
    shape=(TARGET_PASSAGES,),
)
embeddings_mm = np.memmap(
    embeddings_path,
    mode="r",
    dtype=np.float16,
    shape=(TARGET_PASSAGES, DIMENSION),
)

print("Query vectors:", query_vectors.shape)


Batches:   0%|          | 0/14 [00:00<?, ?it/s]

Query vectors: (6980, 384)


In [10]:
# 10. Metric and timing helpers

def evaluate_rankings(indices, qids, doc_ids, qrels, top_k=10):
    """Evaluate strict Recall@K, Success@K, MRR@K, and binary nDCG@K.

    strict Recall@K:
        retrieved judged-relevant passages / all judged-relevant passages.

    Success@K:
        fraction of queries with at least one judged-relevant passage in top K.
        This was previously labeled recall_at_10; it is now reported explicitly.
    """
    strict_recalls, successes, mrrs, ndcgs = [], [], [], []

    for qid, row in zip(qids, indices):
        relevant = qrels[qid]
        ranked_pids = [str(int(doc_ids[idx])) for idx in row if idx >= 0]

        hits = [1 if pid in relevant else 0 for pid in ranked_pids]
        retrieved_relevant = sum(hits)

        strict_recalls.append(retrieved_relevant / len(relevant))
        successes.append(float(retrieved_relevant > 0))

        reciprocal_rank = 0.0
        for rank, hit in enumerate(hits, start=1):
            if hit:
                reciprocal_rank = 1.0 / rank
                break
        mrrs.append(reciprocal_rank)

        dcg = sum(
            hit / math.log2(rank + 1)
            for rank, hit in enumerate(hits, start=1)
        )
        ideal_hits = min(len(relevant), top_k)
        idcg = sum(
            1.0 / math.log2(rank + 1)
            for rank in range(1, ideal_hits + 1)
        )
        ndcgs.append(dcg / idcg if idcg else 0.0)

    return {
        f"recall_at_{top_k}": float(np.mean(strict_recalls)),
        f"success_at_{top_k}": float(np.mean(successes)),
        f"mrr_at_{top_k}": float(np.mean(mrrs)),
        f"ndcg_at_{top_k}": float(np.mean(ndcgs)),
    }


def timed_search(index, vectors, batch_size=64, top_k=10):
    """Run all queries for quality; exclude final partial batch from timing only."""
    all_scores, all_indices = [], []
    per_query_ms = []

    for start in tqdm(range(0, len(vectors), batch_size), desc="Searching"):
        batch = np.ascontiguousarray(
            vectors[start:start + batch_size].astype(np.float32)
        )
        t0 = time.perf_counter()
        scores, indices = index.search(batch, top_k)
        elapsed = time.perf_counter() - t0

        all_scores.append(scores)
        all_indices.append(indices)

        if len(batch) == batch_size:
            per_query_ms.append((elapsed * 1000.0) / batch_size)

    if not per_query_ms:
        raise ValueError("No complete timing batch was available.")

    return (
        np.vstack(all_scores),
        np.vstack(all_indices),
        {
            "p50_ms_per_query": float(np.percentile(per_query_ms, 50)),
            "p95_ms_per_query": float(np.percentile(per_query_ms, 95)),
            "qps": float(1000.0 / np.mean(per_query_ms)),
            "timed_queries": len(per_query_ms) * batch_size,
            "tail_queries_excluded": len(vectors) % batch_size,
        },
    )


def make_gpu_ivfpq(resource, dimension, nlist, m, nbits):
    """Create a T4-compatible GPU IVF-PQ index using FP16 LUTs."""
    gpu_config = faiss.GpuIndexIVFPQConfig()
    gpu_config.device = 0
    gpu_config.useFloat16LookupTables = True

    index = faiss.GpuIndexIVFPQ(
        resource,
        dimension,
        nlist,
        m,
        nbits,
        faiss.METRIC_INNER_PRODUCT,
        gpu_config,
    )
    return index, gpu_config


def stream_add_transformed(index, embeddings, transform=None, batch_size=50_000):
    """Add vectors to an index in batches, optionally through a Faiss transform."""
    for start in tqdm(
        range(0, len(embeddings), batch_size),
        desc="Adding vectors",
    ):
        end = min(start + batch_size, len(embeddings))
        batch = np.ascontiguousarray(embeddings[start:end].astype(np.float32))

        if transform is not None:
            batch = np.ascontiguousarray(transform.apply_py(batch).astype(np.float32))

        index.add(batch)


In [11]:
# 11. Build exact GPU FlatIP baseline

res = faiss.StandardGpuResources()
flat_gpu = faiss.GpuIndexFlatIP(res, DIMENSION)

flat_build_start = time.perf_counter()
for start in tqdm(range(0, TARGET_PASSAGES, ADD_BATCH_SIZE), desc="Adding FlatIP"):
    end = min(start + ADD_BATCH_SIZE, TARGET_PASSAGES)
    batch = np.ascontiguousarray(
        embeddings_mm[start:end].astype(np.float32)
    )
    flat_gpu.add(batch)

flat_build_seconds = time.perf_counter() - flat_build_start

flat_scores, flat_indices, flat_timing = timed_search(
    flat_gpu,
    query_vectors,
    batch_size=SEARCH_BATCH_SIZE,
    top_k=TOP_K,
)
flat_metrics = evaluate_rankings(
    flat_indices,
    eval_qids,
    doc_ids_mm,
    qrels,
    top_k=TOP_K,
)

print("Flat metrics:", flat_metrics)
print("Flat timing:", flat_timing)


Adding FlatIP:   0%|          | 0/20 [00:00<?, ?it/s]

Searching:   0%|          | 0/110 [00:00<?, ?it/s]

Flat metrics: {'recall_at_10': 0.8641833810888252, 'mrr_at_10': 0.6370044799199528, 'ndcg_at_10': 0.6870256121886157}
Flat timing: {'p50_ms_per_query': 0.36540982813448863, 'p95_ms_per_query': 0.3781646749956735, 'qps': 2739.4944575727563, 'timed_queries': 6976, 'tail_queries_excluded': 4}


In [13]:
# 12. Build GPU IVF-PQ M=96 baseline in streaming add batches

train_rng = np.random.default_rng(SEED)
train_indices = train_rng.choice(
    TARGET_PASSAGES,
    size=min(TRAIN_SAMPLE_SIZE, TARGET_PASSAGES),
    replace=False,
)
train_vectors = np.ascontiguousarray(
    embeddings_mm[train_indices].astype(np.float32)
)

ivfpq_gpu, gpu_config = make_gpu_ivfpq(
    res,
    DIMENSION,
    NLIST,
    M,
    NBITS,
)

print(
    f"Training GPU IVF-PQ: nlist={NLIST}, M={M}, nbits={NBITS}, "
    f"train_vectors={len(train_vectors):,}, "
    f"FP16_LUT={gpu_config.useFloat16LookupTables}"
)

plain_build_start = time.perf_counter()
ivfpq_gpu.train(train_vectors)
stream_add_transformed(
    ivfpq_gpu,
    embeddings_mm,
    transform=None,
    batch_size=ADD_BATCH_SIZE,
)
plain_build_seconds = time.perf_counter() - plain_build_start

print(
    f"Plain IVF-PQ complete: {ivfpq_gpu.ntotal:,} vectors indexed in "
    f"{plain_build_seconds / 60:.2f} minutes"
)

plain_results = []
for nprobe in NPROBE_VALUES:
    ivfpq_gpu.nprobe = nprobe

    scores, indices, timing = timed_search(
        ivfpq_gpu,
        query_vectors,
        batch_size=SEARCH_BATCH_SIZE,
        top_k=TOP_K,
    )
    metrics = evaluate_rankings(
        indices,
        eval_qids,
        doc_ids_mm,
        qrels,
        top_k=TOP_K,
    )

    plain_results.append(
        {
            "method": "GPU IVF-PQ M=96 FP16-LUT",
            "nprobe": nprobe,
            "lookup_table_precision": "float16",
            "index_vectors": int(ivfpq_gpu.ntotal),
            **metrics,
            **timing,
            "build_seconds": plain_build_seconds,
        }
    )

# Serialize the CPU index for storage accounting.
plain_ivfpq_cpu = faiss.index_gpu_to_cpu(ivfpq_gpu)
plain_index_path = RESULT_DIR / (
    f"msmarco_{TARGET_PASSAGES // 1_000_000}m_ivfpq_m{M}_fp16lut.faiss"
)
faiss.write_index(plain_ivfpq_cpu, str(plain_index_path))

plain_index_bytes = plain_index_path.stat().st_size

print(
    "Serialized plain IVF-PQ index:",
    plain_index_path,
    f"{plain_index_bytes / 1e9:.3f} GB",
)
display(pd.DataFrame(plain_results))


Training GPU IVF-PQ: nlist=4096, M=96, nbits=8, train_vectors=200,000, FP16_LUT=True


Adding IVF-PQ:   0%|          | 0/20 [00:00<?, ?it/s]

IVF-PQ build complete: 1,000,000 vectors indexed in 0.28 minutes


Searching:   0%|          | 0/110 [00:00<?, ?it/s]

Searching:   0%|          | 0/110 [00:00<?, ?it/s]

Serialized IVF-PQ index: msmarco_scale_results/msmarco_1m_ivfpq_m96_fp16lut.faiss 0.111 GB


,method,nprobe,lookup_table_precision,index_vectors,recall_at_10,mrr_at_10,ndcg_at_10,p50_ms_per_query,p95_ms_per_query,qps,timed_queries,tail_queries_excluded,build_seconds
0,GPU IVF-PQ M=96 FP16-LUT,16,float16,1000000,0.728797,0.536127,0.576951,0.021331,0.034659,28935.087491,6976,4,16.982003
1,GPU IVF-PQ M=96 FP16-LUT,64,float16,1000000,0.797278,0.583356,0.629670,0.067536,0.079502,14569.386799,6976,4,16.982003


In [ ]:
# 13. Native Faiss OPQMatrix + GPU IVF-PQ baseline

# OPQ is trained natively by Faiss on the same deterministic sample.
# The transform is then applied to document and query vectors before IVF-PQ.
opq = faiss.OPQMatrix(DIMENSION, M)
opq.niter = OPQ_NITER
opq.niter_pq = OPQ_NITER_PQ
opq.verbose = True

print(
    f"Training native Faiss OPQMatrix: M={M}, "
    f"niter={OPQ_NITER}, niter_pq={OPQ_NITER_PQ}"
)

opq_train_start = time.perf_counter()
opq.train(train_vectors)

opq_train_vectors = np.ascontiguousarray(
    opq.apply_py(train_vectors).astype(np.float32)
)

opq_ivfpq_gpu, opq_gpu_config = make_gpu_ivfpq(
    res,
    DIMENSION,
    NLIST,
    M,
    NBITS,
)

opq_ivfpq_gpu.train(opq_train_vectors)
stream_add_transformed(
    opq_ivfpq_gpu,
    embeddings_mm,
    transform=opq,
    batch_size=ADD_BATCH_SIZE,
)

opq_build_seconds = time.perf_counter() - opq_train_start

query_vectors_opq = np.ascontiguousarray(
    opq.apply_py(query_vectors).astype(np.float32)
)

print(
    f"Native OPQ-IVF-PQ complete: {opq_ivfpq_gpu.ntotal:,} vectors indexed in "
    f"{opq_build_seconds / 60:.2f} minutes"
)

opq_results = []
for nprobe in NPROBE_VALUES:
    opq_ivfpq_gpu.nprobe = nprobe

    scores, indices, timing = timed_search(
        opq_ivfpq_gpu,
        query_vectors_opq,
        batch_size=SEARCH_BATCH_SIZE,
        top_k=TOP_K,
    )
    metrics = evaluate_rankings(
        indices,
        eval_qids,
        doc_ids_mm,
        qrels,
        top_k=TOP_K,
    )

    opq_results.append(
        {
            "method": "Native Faiss OPQMatrix + GPU IVF-PQ M=96 FP16-LUT",
            "nprobe": nprobe,
            "lookup_table_precision": "float16",
            "index_vectors": int(opq_ivfpq_gpu.ntotal),
            **metrics,
            **timing,
            "build_seconds": opq_build_seconds,
        }
    )

# Deployment-aware storage: serialized GPU-converted IVF-PQ index plus the
# externally stored 384x384 FP32 OPQ rotation matrix required at query time.
opq_ivfpq_cpu = faiss.index_gpu_to_cpu(opq_ivfpq_gpu)
opq_index_path = RESULT_DIR / (
    f"msmarco_{TARGET_PASSAGES // 1_000_000}m_native_opq_ivfpq_m{M}_fp16lut.faiss"
)
faiss.write_index(opq_ivfpq_cpu, str(opq_index_path))

opq_rotation = faiss.vector_to_array(opq.A).reshape(DIMENSION, DIMENSION)
opq_rotation_path = RESULT_DIR / (
    f"msmarco_{TARGET_PASSAGES // 1_000_000}m_native_opq_rotation.npy"
)
np.save(opq_rotation_path, opq_rotation.astype(np.float32))

opq_index_bytes = opq_index_path.stat().st_size
opq_rotation_bytes = opq_rotation_path.stat().st_size

print(
    "Serialized native OPQ IVF-PQ index:",
    opq_index_path,
    f"{opq_index_bytes / 1e9:.3f} GB",
)
print(
    "Serialized OPQ rotation:",
    opq_rotation_path,
    f"{opq_rotation_bytes / 1e6:.3f} MB",
)

# Training sample is no longer needed after both baselines have been built.
del train_vectors, opq_train_vectors
gc.collect()
torch.cuda.empty_cache()

display(pd.DataFrame(opq_results))


In [ ]:
# 14. Consolidate benchmark results and write reproducible outputs

float32_vector_bytes = TARGET_PASSAGES * DIMENSION * 4
ids_bytes = TARGET_PASSAGES * 8
float32_deployment_bytes = float32_vector_bytes + ids_bytes

plain_deployment_bytes = plain_index_bytes + ids_bytes
opq_deployment_bytes = opq_index_bytes + opq_rotation_bytes + ids_bytes

summary_rows = [{
    "method": "GPU FlatIP",
    "nprobe": None,
    **flat_metrics,
    **flat_timing,
    "build_seconds": flat_build_seconds,
    "serialized_index_bytes": float32_vector_bytes,
    "external_transform_bytes": 0,
    "serialized_total_bytes": float32_deployment_bytes,
    "serialized_compression_x": 1.0,
}]

for row in plain_results:
    summary_rows.append({
        **row,
        "serialized_index_bytes": plain_index_bytes,
        "external_transform_bytes": 0,
        "serialized_total_bytes": plain_deployment_bytes,
        "serialized_compression_x": float32_deployment_bytes / plain_deployment_bytes,
    })

for row in opq_results:
    summary_rows.append({
        **row,
        "serialized_index_bytes": opq_index_bytes,
        "external_transform_bytes": opq_rotation_bytes,
        "serialized_total_bytes": opq_deployment_bytes,
        "serialized_compression_x": float32_deployment_bytes / opq_deployment_bytes,
    })

summary = pd.DataFrame(summary_rows).sort_values(
    ["method", "nprobe"],
    na_position="first",
).reset_index(drop=True)

summary_path = RESULT_DIR / (
    f"msmarco_{TARGET_PASSAGES // 1_000_000}m_ivfpq_opq_summary.csv"
)
summary.to_csv(summary_path, index=False)

metadata = {
    "dataset": "MS MARCO passage ranking",
    "target_passages": TARGET_PASSAGES,
    "sampling": "qrels-preserving random distractor subset",
    "embedding_model": EMBEDDING_MODEL,
    "embedding_dimension": DIMENSION,
    "embedding_storage_dtype": "float16 memmap; converted to float32 for Faiss train/add",
    "nlist": NLIST,
    "M": M,
    "nbits": NBITS,
    "nprobe_values": NPROBE_VALUES,
    "top_k": TOP_K,
    "metric_definitions": {
        "recall_at_10": "retrieved judged-relevant passages / all judged-relevant passages",
        "success_at_10": "fraction of queries with at least one judged-relevant passage in top 10",
        "mrr_at_10": "mean reciprocal rank truncated at 10",
        "ndcg_at_10": "binary relevance nDCG truncated at 10",
    },
    "opq": {
        "implementation": "Faiss OPQMatrix",
        "niter": OPQ_NITER,
        "niter_pq": OPQ_NITER_PQ,
        "rotation_storage": "384x384 float32 .npy",
    },
    "gpu": torch.cuda.get_device_name(0),
    "ram_gb": round(psutil.virtual_memory().total / (1024**3), 2),
}
metadata_path = RESULT_DIR / (
    f"msmarco_{TARGET_PASSAGES // 1_000_000}m_ivfpq_opq_metadata.json"
)
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

display(summary)
print("Saved summary:", summary_path)
print("Saved metadata:", metadata_path)


## Phase 1.1 completion criteria

Before increasing scale:

1. Verify that both plain IVF-PQ and native Faiss OPQMatrix + IVF-PQ complete without OOM.
2. Report **both strict Recall@10 and Success@10**; do not call Success@10 “Recall@10”.
3. Compare `nprobe = 8, 16, 32, 64` on quality, P95, and QPS.
4. Keep the OPQ rotation file in serialized deployment accounting.
5. Inspect at least a few query results manually before publishing the table.
6. Only after 1M results are stable, extend the same protocol to 2M and 4M.

The next research step after this native Faiss baseline is a matched custom PyTorch OPQ run, followed by an explicit comparison of quality, build cost, and serving artifact requirements.
